# ema-first-moment composite — cx27: m buffer updated via in-place copy_ (no rebinding)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `ema-first-moment`, `buffer-copy_-inplace`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "ema-first-moment"
DD_ATOM_IDS = ["ema-first-moment", "buffer-copy_-inplace"]
DD_SUBTOPICS = ["Optimizer: Adam EMA first moment", "PyTorch: in-place buffer copy"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Adam's first-moment `m` is a per-parameter buffer — a running EMA of the GRADIENTS (not the squared gradients; that's `v`). The same buffer-aliasing trap applies: rebinding (`m = beta1*m + ...`) orphans the registered buffer; `m.copy_(...)` writes back into the same storage so anyone holding the old reference sees the update.

**The two atoms.**
- **ema-first-moment** — `m_new = beta1 * m + (1 - beta1) * g`.
- **buffer-copy_-inplace** — `m.copy_(m_new)`.

**Why care about this pattern.** Adam state buffers live in the optimizer's `state[param]` dict, keyed by parameter identity. If you rebind, the dict still points at the old, stale tensor — the optimizer effectively starts every step from `m=0` again. The bug is silent because the formula still 'runs'; only the convergence behaviour looks subtly off.

### Composite Exercise — m buffer updated via in-place copy_ (no rebinding)

**Atoms exercised together**: `ema-first-moment`, `buffer-copy_-inplace`

Implement `cx27_update_m_via_copy(m, g, beta1)`.

Required behaviour:
1. Compute `m_new = beta1 * m + (1 - beta1) * g` (atom: ema-first-moment).
2. Write back via `m.copy_(m_new)` (atom: buffer-copy_-inplace).
3. Return `None`. `id(m)` and `m.data_ptr()` MUST be unchanged.

The test checks: in-place semantics, numerical correctness over a few steps, robustness when `g.requires_grad is True`, and aliasing (a dict reference held to `m` must see the update).

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx27_update_m_via_copy(m, g, beta1):
    """In-place update m <- beta1*m + (1-beta1)*g via copy_. Returns None."""
    raise NotImplementedError

def _test_cx27():
    # Case A: in-place semantics.
    t.manual_seed(0)
    m = t.randn(3, 4)
    g = t.randn(3, 4)
    m_id_before = id(m)
    m_ptr_before = m.data_ptr()
    m_before = m.clone()
    beta1 = 0.9
    ret = cx27_update_m_via_copy(m, g, beta1)
    assert ret is None
    assert id(m) == m_id_before, 'm was rebound'
    assert m.data_ptr() == m_ptr_before, 'm storage changed — copy_ preserves data_ptr'
    expected = beta1 * m_before + (1 - beta1) * g
    assert t.allclose(m, expected, atol=1e-7)

    # Case B: dict-aliased reference sees the update.
    m2 = t.zeros(5)
    stash = {'m_ref': m2}
    g2 = t.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
    cx27_update_m_via_copy(m2, g2, beta1=0.5)
    # m1 = 0.5*0 + 0.5*g = [0.5, 1.0, 1.5, 2.0, 2.5].
    assert t.allclose(stash['m_ref'], t.tensor([0.5, 1.0, 1.5, 2.0, 2.5]), atol=1e-7), (
        'aliased reference did not see the update'
    )

    # Case C: works with grad-requiring g.
    m3 = t.ones(4)
    g3 = t.tensor([0.1, 0.2, 0.3, 0.4], requires_grad=True)
    cx27_update_m_via_copy(m3, g3, beta1=0.9)
    expected3 = 0.9 * t.ones(4) + 0.1 * g3.detach()
    assert t.allclose(m3, expected3, atol=1e-7)
    assert m3.requires_grad is False

    # Case D: multi-step accumulation.
    m4 = t.zeros(2)
    for step in range(3):
        g_step = t.tensor([1.0, -1.0])
        cx27_update_m_via_copy(m4, g_step, beta1=0.5)
    # m0=0, m1=0.5*g, m2=0.5*0.5*g+0.5*g=0.75*g, m3=0.5*0.75*g+0.5*g=0.875*g.
    assert t.allclose(m4, 0.875 * t.tensor([1.0, -1.0]), atol=1e-7), (
        f'multi-step EMA broken; got {m4.tolist()}'
    )
    _dd_passed.add('cx27')

_test_cx27()

<details><summary>Show solution — cx27</summary>

```python
def cx27_update_m_via_copy(m, g, beta1):
    with t.no_grad():
        # Atom A (ema-first-moment): m_new = beta1*m + (1-beta1)*g.
        m_new = beta1 * m + (1 - beta1) * g
        # Atom B (buffer-copy_-inplace): preserve storage identity.
        m.copy_(m_new)
```

The contrast with cx26 is just `g` vs `g*g` — the structural lesson (use `copy_` to keep the buffer aliased) is identical. In real Adam, `m` is also commonly updated via the fused `m.mul_(beta1).add_(g, alpha=1-beta1)` for one less temporary allocation.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx27'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx27',
        'subtopics': ["Optimizer: Adam EMA first moment", "PyTorch: in-place buffer copy"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()